# Import required libraries and mount Google Drive

In [ ]:
# notebooks/01_Data_Preparation.ipynb

import os
import zipfile
import json
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
from google.colab import drive
from tqdm import tqdm

# Mount Google Drive to access persistent storage
drive.mount('/content/drive')

# Add src to path
PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)
import src.config as config

Mounted at /content/drive


# Extract dataset zip file to local storage

In [ ]:
print("Extracting dataset zip file to local runtime storage...")
os.makedirs(config.LOCAL_EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(config.ZIP_SOURCE_PATH, 'r') as zip_ref:
    unzip_targets = zip_ref.namelist()
    for file in tqdm(unzip_targets, desc="Extracting dataset"):
        zip_ref.extract(file, config.LOCAL_EXTRACT_DIR)

print(f"Extraction complete. Data directory: {config.DATA_ROOT_DIR}")

Extracting dataset zip file to local runtime storage...


Extracting dataset: 100%|██████████| 1847/1847 [05:14<00:00,  5.87it/s]

Extraction complete. Data directory: /content/MICCAI_BraTS2020_TrainingData


# Parse name_mapping.csv and prepare patient dictionaries

In [ ]:
print("Parsing metadata and creating stratified splits...")
df = pd.read_csv(config.NAME_MAPPING_CSV)

# Map grades to binary integers (0: LGG, 1: HGG)
grade_map = {"LGG": 0, "HGG": 1}
df['grade_int'] = df['Grade'].map(grade_map)

patient_records = []
for idx, row in df.iterrows():
    subject_id = row['BraTS_2020_subject_ID']
    grade = row['grade_int']

    # Paths for each structural modality matching the specified sequence order
    modality_paths = [
        os.path.join(config.DATA_ROOT_DIR, subject_id, f"{subject_id}_{mod}.nii")
        for mod in config.MODALITIES
    ]

    # Path for the segmentation target mask
    seg_path = os.path.join(config.DATA_ROOT_DIR, subject_id, f"{subject_id}_seg.nii")

    # Check all 5 files exist before adding to records
    files_exist = all(os.path.exists(p) for p in modality_paths) and os.path.exists(seg_path)
    if not files_exist:
        print(f"Warning: Missing files for subject {subject_id}. Skipping.")
        continue

    patient_records.append({
        "image": modality_paths,
        "label": seg_path,
        "grade": int(grade)
    })

Parsing metadata and creating stratified splits...


# Execute stratified data splitting to preserve HGG/LGG distribution ratio

In [ ]:
grades = [record['grade'] for record in patient_records]

# Separate into 80% train and 20% val/test combined
train_records, val_test_records = train_test_split(
    patient_records,
    test_size=(config.VAL_RATIO + config.TEST_RATIO),
    stratify=grades,
    random_state=config.RANDOM_SEED
)

# Split the remaining 20% equally into vali (10%) and test (10%)
val_test_grades = [record['grade'] for record in val_test_records]
val_records, test_records = train_test_split(
    val_test_records,
    test_size=0.5,
    stratify=val_test_grades,
    random_state=config.RANDOM_SEED
)

print(f"Split distribution summary:")
print(f"Train samples: {len(train_records)}")
print(f"Validation samples: {len(val_records)}")
print(f"Test samples: {len(test_records)}")

Split distribution summary:
Train samples: 294
Validation samples: 37
Test samples: 37


# Save dataset split configuration to a single JSON file

In [ ]:
split_data = {
    "train": train_records,
    "val": val_records,
    "test": test_records
}

os.makedirs(os.path.dirname(config.DATA_SPLIT_JSON), exist_ok=True)

with open(config.DATA_SPLIT_JSON, 'w') as f:
    json.dump(split_data, f, indent=4)

print(f"Successfully saved data splits configuration to {config.DATA_SPLIT_JSON}")

Successfully saved data splits configuration to /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/data_splits.json
